# 03 -- Security-Domain Foundation Model: Loading and First Prompts

**Goal:** load a domain-specialized security LLM (Cisco Foundation AI's `Foundation-Sec-1.1-8B-Instruct`) locally on this Mac, confirm it runs, and try it on three simple SOC-relevant prompts.

This is a **different kind of model** than `00`-`02`: CTSM is a numeric time-series forecaster; this is a **text/language model** fine-tuned on cybersecurity content. Same lab, same reused Python 3.11 environment, genuinely different modality.

**Scope for this notebook -- deliberately narrow:**
1. Verify the environment.
2. Load the quantized model and confirm inference works.
3. Run 3 simple security-analysis prompts and look at the raw responses.

**Explicitly out of scope here** (saved for follow-up notebooks): scoring the responses for quality/hallucination, comparing against a general-purpose (non-security) model, retrieval-augmented generation (RAG), agentic orchestration, and any fine-tuning. This notebook only answers "does it load and run, and what does it sound like" -- nothing more.

## Why this model needed a different loading approach than CTSM

`Foundation-Sec-1.1-8B-Instruct` is an 8-billion-parameter Llama-3.1-based model. Loaded at full precision (bf16, the format Cisco published it in), it needs about **16GB just for the weights** -- on a 16GB unified-memory Mac, that leaves nothing for macOS, Jupyter, or anything else. CTSM's 250M parameters never had this problem; an 8B-parameter language model does.

The fix is **quantization**: compressing each weight from 16 bits down to about 4-5 bits, trading a small amount of precision for a large reduction in memory. Cisco publishes pre-quantized **GGUF** files for exactly this purpose -- GGUF is a single-file format designed for `llama.cpp`, the inference engine used here (via its Python bindings, `llama-cpp-python`), rather than the `transformers`/PyTorch path CTSM used. We're loading the **Q4_K_M** quantization: `foundation-sec-1.1-8b-instruct-q4_k_m.gguf`, about 4.92GB, from [fdtn-ai/Foundation-Sec-1.1-8B-Instruct-Q4_K_M-GGUF](https://huggingface.co/fdtn-ai/Foundation-Sec-1.1-8B-Instruct-Q4_K_M-GGUF) on Hugging Face.

Two other approaches were considered and ruled out:
- **PyTorch + MPS** (Apple's Metal backend, same as CTSM used): no reliable 4-bit quantization path exists there today -- `bitsandbytes`, the usual quantization library, is CUDA-only. Loading bf16 directly would need the full ~16GB.
- **Ollama**: also uses GGUF under the hood, but runs as a separate background service you talk to over HTTP, rather than loading directly into this notebook's Python process.

`llama-cpp-python` was chosen because it's the lightest path that (a) fits in 16GB via quantization and (b) loads the model directly into this notebook, the same self-contained pattern as CTSM.

**Licensing note:** this model is not fully open-source. Per its `NOTICE.md`, the underlying Llama 3.1 weights are governed by Meta's Llama 3.1 Community License, and Cisco's changes on top are Apache-2.0. Fine for local, non-commercial, personal learning use like this notebook.

## Step 0 -- Verify the environment

Same check as `00_verify_setup.ipynb`: confirm this cell is actually running inside the project's existing `.venv` on Python 3.11 -- no new environment was created for this notebook.

In [1]:
import sys, platform
from pathlib import Path

exe_parts = Path(sys.executable).parts
shown_exe = str(Path(*exe_parts[exe_parts.index(".venv"):])) if ".venv" in exe_parts else sys.executable

print("Python executable:", shown_exe)
print("Python version   :", sys.version.split()[0])
print("Platform         :", platform.platform())

assert sys.version_info[:2] == (3, 11), "Expected Python 3.11 -- wrong kernel selected?"
assert ".venv" in sys.executable and "ai-foundation-models-lab" in sys.executable, (
    "Interpreter is not this project's venv."
)
print("\nOK: same project venv as notebooks 00-02, no new environment created.")

Python executable: .venv/bin/python
Python version   : 3.11.15
Platform         : macOS-26.5-arm64-arm-64bit

OK: same project venv as notebooks 00-02, no new environment created.


## Step 1 -- Download and load the quantized model

`hf_hub_download` fetches the GGUF file from Hugging Face into the shared local cache (`~/.cache/huggingface`, the same cache CTSM's weights already live in) if it isn't there already -- so re-running this cell later is instant and offline.

Then `Llama(...)` loads it into memory:
- `n_gpu_layers=-1` -- offload every transformer layer to the GPU via Metal, rather than running on CPU. On Apple Silicon's unified memory, "GPU" and "CPU" memory are the same pool, so this is purely a speed optimization, not a memory tradeoff.
- `n_ctx=2048` -- the context window we're allocating for this session, in tokens. The model itself supports up to 64k tokens, but our prompts here are short and single-turn, so a small context keeps the KV cache (the model's working memory for the current conversation) tiny. We'll revisit larger context sizes when a future notebook needs to feed it longer documents.

In [2]:
import warnings
warnings.filterwarnings("ignore", message="IProgress not found")

import time
from pathlib import Path
from huggingface_hub import hf_hub_download
from llama_cpp import Llama

REPO_ID = "fdtn-ai/Foundation-Sec-1.1-8B-Instruct-Q4_K_M-GGUF"
FILENAME = "foundation-sec-1.1-8b-instruct-q4_k_m.gguf"

t0 = time.time()
model_path = hf_hub_download(repo_id=REPO_ID, filename=FILENAME)
print(f"Model file: {FILENAME} ({Path(model_path).stat().st_size / 1e9:.2f} GB)")
print(f"Resolved in {time.time() - t0:.1f}s (instant + offline if already cached)")

t0 = time.time()
llm = Llama(
    model_path=model_path,
    n_ctx=2048,
    n_gpu_layers=-1,
    verbose=False,
)
print(f"\nModel loaded in {time.time() - t0:.1f}s")

Model file: foundation-sec-1.1-8b-instruct-q4_k_m.gguf (4.92 GB)
Resolved in 0.2s (instant + offline if already cached)



Model loaded in 1.1s


## Step 2 -- Confirm inference actually works

One trivial, low-stakes prompt before the real experiment -- the same instinct as `00_verify_setup.ipynb`'s smoke test for CTSM. We use `create_chat_completion` rather than raw text completion because this is an **instruction-tuned** model: it was fine-tuned to expect input wrapped in a specific chat template (system/user/assistant role tags matching Llama-3.1's format), and `create_chat_completion` builds that template for us automatically from a plain `role`/`content` message list.

`temperature=0.2` keeps sampling close to deterministic -- appropriate for an analysis task where we want a consistent, literal answer rather than creative variation.

In [3]:
def ask(prompt, max_tokens=200, temperature=0.2):
    response = llm.create_chat_completion(
        messages=[{"role": "user", "content": prompt}],
        max_tokens=max_tokens,
        temperature=temperature,
    )
    return response["choices"][0]["message"]["content"].strip()

sanity_check = ask("In one sentence, what does a SOC analyst do?", max_tokens=60)
print(sanity_check)
print("\nInference pipeline confirmed working.")

A SOC analyst monitors and responds to security incidents to protect an organization's digital assets.

Inference pipeline confirmed working.


## Step 3 -- Three simple security-analysis prompts

Three short, self-contained scenarios, each probing a different capability the model card claims under "SOC Acceleration":

1. **Triage/summarization** -- given a raw log pattern, explain what it suggests and how urgent it is. Tests whether the model recognizes a common attack signature (brute-force login) from unstructured text.
2. **Classification against a known framework** -- map an observed behavior to a MITRE ATT&CK technique. Tests recall of domain-specific taxonomy that a general-purpose model wouldn't have been trained to prioritize.
3. **Recommended actions** -- given a confirmed incident, list immediate containment steps. Tests applied reasoning, not just recall.

These are illustrative, not a benchmark -- three prompts, one run each, no scoring. Just look at what comes back.

In [4]:
prompts = {
    "1. Triage / summarization": (
        "A SOC analyst sees this log line from a Linux server, repeated 47 times in 2 minutes:\n"
        "'Failed password for invalid user admin from 203.0.113.45 port 51422 ssh2'\n"
        "It is immediately followed by one successful login as 'admin' from the same IP.\n"
        "In 3-4 sentences, explain what this suggests and how urgent it is."
    ),
    "2. MITRE ATT&CK mapping": (
        "Map the following observed behavior to the most likely MITRE ATT&CK technique "
        "(give the technique ID and name): an attacker used a valid stolen VPN credential to log in, "
        "then ran 'whoami', 'net user', and 'net group \"Domain Admins\"' shortly after gaining access."
    ),
    "3. Recommended containment actions": (
        "A SOC analyst confirms a workstation is beaconing to an external IP every 60 seconds over port 443, "
        "with encrypted traffic that does not match normal browser TLS patterns. "
        "List the top 3 immediate containment actions, in priority order."
    ),
}

responses = {}
for label, prompt in prompts.items():
    responses[label] = ask(prompt, max_tokens=250)

## Step 4 -- Display the responses

In [5]:
for label, response in responses.items():
    print(f"=== {label} ===")
    print(f"Prompt: {prompts[label]}\n")
    print(f"Response:\n{response}")
    print("\n" + "-" * 80 + "\n")

=== 1. Triage / summarization ===
Prompt: A SOC analyst sees this log line from a Linux server, repeated 47 times in 2 minutes:
'Failed password for invalid user admin from 203.0.113.45 port 51422 ssh2'
It is immediately followed by one successful login as 'admin' from the same IP.
In 3-4 sentences, explain what this suggests and how urgent it is.

Response:
A SOC analyst sees this log line indicating repeated failed login attempts for an invalid user ('admin') from a specific IP address (203.0.113.45). This suggests a potential brute force attack aimed at gaining unauthorized access to the server. The urgency is high because such attacks can lead to unauthorized access, data theft, or system compromise if successful. Immediate investigation and response are necessary to mitigate the risk and prevent potential damage.

--------------------------------------------------------------------------------

=== 2. MITRE ATT&CK mapping ===
Prompt: Map the following observed behavior to the most

## Summary and stopping point

In this notebook we: confirmed the environment, loaded a quantized 8B-parameter security-domain LLM entirely locally on 16GB of unified memory, verified inference works, and ran it on three simple SOC-style prompts -- all without writing a single line of training code.

It's worth naming where this sits in the foundation-model lifecycle, since that framing is the point of this whole lab: `Foundation-Sec-1.1-8B-Instruct` is the product of **pretraining** (Meta's Llama-3.1-8B, on general text), then **continued pretraining** (Cisco's Foundation AI team, on cybersecurity-specific text -- vulnerability databases, MITRE ATT&CK, threat intel), then **instruction tuning / alignment** (turning it into something that follows chat-style instructions), then **quantization** (compressing it to GGUF Q4_K_M for local deployment), and finally **inference** -- which is all we did here.

Deliberately not done yet, and each a distinct next step:
- **Evaluation** -- actually scoring these responses for correctness, hallucination, and evidence quality, and running the same prompts through a general-purpose (non-security) model for comparison.
- **RAG** -- grounding responses in real, retrieved documents (e.g. an actual playbook or threat intel feed) instead of relying purely on what the model memorized during training.
- **Agentic orchestration** -- letting the model take multi-step actions (look something up, call a tool, revise its own plan) rather than answering a single prompt once.
- **Fine-tuning** -- not attempted at all in this lab yet.

Each of those is a meaningfully different, larger piece of work, which is exactly why they're separate notebooks rather than more cells bolted onto this one.